In [ ]:
# =================================================================
# Diving Deep into Vector Databases and RAG Chatbots
# =================================================================

# Data Loading And Preparation
print("=" * 60)
print("EXERCICE 1: Data Loading And Preparation")
print("=" * 60)

# 1. Installation des bibliothèques requises
!pip install -q faiss-cpu==1.7.4
!pip install -q chromadb==0.3.21
!pip install -qU chromadb
!pip install -q numpy<2
!pip install -q sentence-transformers
!pip install -q transformers
!mkdir -p cache

# 2. Imports
import numpy as np
import pandas as pd
import faiss
import json
from sentence_transformers import SentenceTransformer, InputExample
import chromadb
from chromadb.config import Settings
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

# 3. Chargement du dataset
path = "https://github.com/devtlv/Datasets-GEN-AI-Bootcamp/raw/refs/heads/main/Week%205/Day%204%20-%20Diving%20Deep%20into%20Vector%20Databases%20and%20RAG%20Chatbots/labelled_newscatcher_dataset.zip"
pdf = pd.read_csv(path, compression='zip')

# 4. Ajout d'une colonne d'identifiant
pdf["id"] = range(len(pdf))

# 5. Inspection des données
print("\nAperçu du dataset:")
display(pdf.head())
print(f"\nNombre de lignes: {len(pdf)}")
print(f"Colonnes: {pdf.columns.tolist()}")

# 6. Création d'un subset pour un traitement plus rapide
pdf_subset = pdf.head(1000)
print(f"\nSubset créé avec {len(pdf_subset)} lignes")

# =================================================================
print("\n" + "=" * 60)
print("EXERCICE 2: Vectorization With Sentence Transformers")
print("=" * 60)

# 1. Fonction helper pour créer des InputExamples
def example_create_fn(doc1: pd.Series) -> InputExample:
    """
    Fonction helper qui génère un InputExample pour sentence_transformer
    """
    return InputExample(texts=[doc1])

# 2. Application de la fonction au subset
faiss_train_examples = pdf_subset.apply(
    lambda x: example_create_fn(x["title"]), axis=1
).tolist()
print(f"\nNombre d'exemples créés: {len(faiss_train_examples)}")
print(f"Premiers exemples: {faiss_train_examples[:3]}")

# 3. Initialisation du modèle d'embedding
print("\nChargement du modèle SentenceTransformer...")
model = SentenceTransformer('all-MiniLM-L6-v2')

# 4. Extraction des titres
titles_list = pdf_subset["title"].tolist()
print(f"\nNombre de titres à vectoriser: {len(titles_list)}")

# 5. Génération des embeddings
print("\nGénération des embeddings...")
faiss_title_embedding = model.encode(titles_list)

# 6. Vérification des dimensions
print(f"\nDimensions des embeddings:")
print(f"  - Nombre d'embeddings: {len(faiss_title_embedding)}")
print(f"  - Dimension de chaque vecteur: {len(faiss_title_embedding[0])}")

# =================================================================
print("\n" + "=" * 60)
print("EXERCICE 3: FAISS Indexing And Search")
print("=" * 60)

# 1. Préparation des données pour l'indexation
pdf_to_index = pdf_subset
id_index = np.array(pdf_subset["id"].values, dtype=np.int64)

# 2. Normalisation des vecteurs d'embedding
content_encoded_normalized = faiss_title_embedding.astype('float32')
faiss.normalize_L2(content_encoded_normalized)
print(f"\nVecteurs normalisés: {content_encoded_normalized.shape}")

# 3. Création de l'index FAISS
index_content = faiss.IndexIDMap(faiss.IndexFlatIP(len(faiss_title_embedding[0])))
index_content.add_with_ids(content_encoded_normalized, id_index)
print(f"\nIndex FAISS créé avec {index_content.ntotal} vecteurs")

# 4. Fonction de recherche
def search_content(query, pdf_to_index, k=3):
    """
    Recherche les k articles les plus similaires à la requête
    """
    # Encodage de la requête
    query_vector = model.encode([query]).astype('float32')
    faiss.normalize_L2(query_vector)

    # Recherche dans l'index
    top_k = index_content.search(query_vector, k)
    ids = top_k[1][0]
    similarities = top_k[0][0]

    # Récupération des résultats
    results = pdf_to_index[pdf_to_index["id"].isin(ids)]
    results = results.copy()
    results["similarities"] = similarities

    return results

# 5. Test de la fonction de recherche
print("\nRecherche pour le terme 'animal':")
results_animal = search_content("animal", pdf_to_index, k=5)
display(results_animal[["title", "topic", "similarities"]])

# =================================================================
print("\n" + "=" * 60)
print("EXERCICE 4: ChromaDB Collection And Querying")
print("=" * 60)

# 1. Initialisation du client ChromaDB
chroma_client = chromadb.Client()
collection_name = "my_news"

# 2. Suppression de la collection si elle existe déjà
if len(chroma_client.list_collections()) > 0 and collection_name in [chroma_client.list_collections()[0].name]:
    chroma_client.delete_collection(name=collection_name)
    print(f"Collection '{collection_name}' supprimée")

# 3. Création de la collection
print(f"\nCréation de la collection: '{collection_name}'")
collection = chroma_client.create_collection(name=collection_name)

# 4. Ajout des données à la collection
print("\nAjout des 100 premiers articles à ChromaDB...")
display(pdf_subset.head())

collection.add(
    documents=pdf_subset["title"][:100].tolist(),
    metadatas=[{"topic": topic} for topic in pdf_subset["topic"][:100].tolist()],
    ids=[str(i) for i in range(100)]
)
print("Données ajoutées avec succès!")

# 5. Requête de la collection
print("\nRecherche dans ChromaDB pour 'space':")
results = collection.query(
    query_texts=["space"],
    n_results=10
)
print(json.dumps(results, indent=4))

# =================================================================
print("\n" + "=" * 60)
print("EXERCICE 5: Question Answering With Hugging Face Model")
print("=" * 60)

# 1. Initialisation du modèle et tokenizer
print("\nChargement du modèle GPT-2 pour la génération de texte...")
model_id = "gpt2"
tokenizer = AutoTokenizer.from_pretrained(model_id)
lm_model = AutoModelForCausalLM.from_pretrained(model_id)

# 2. Création du pipeline de génération
pipe = pipeline(
    "text-generation",
    model=lm_model,
    tokenizer=tokenizer,
    max_new_tokens=512,
    device_map="auto"
)

# 3. Construction du prompt
question = "What's the latest news on space development?"
context = " ".join([f"#{str(i)}" for i in results["documents"][0]])
prompt_template = f"""Relevant context: {context}

The user's question: {question}

Answer:"""

print(f"\nQuestion: {question}")
print(f"\nContexte récupéré de ChromaDB: {context[:200]}...")

# 4. Génération de la réponse
print("\nGénération de la réponse...")
lm_response = pipe(prompt_template)
print("\n" + "=" * 60)
print("RÉPONSE GÉNÉRÉE:")
print("=" * 60)
print(lm_response[0]["generated_text"])

# =================================================================
print("\n" + "=" * 60)
print("TOUS LES EXERCICES COMPLÉTÉS AVEC SUCCÈS!")
print("=" * 60)